# ETL — Germany Train Data

Ce notebook transforme les données GTFS allemandes en un format standardisé avec les colonnes :
`data_source`, `route_id`, `id_origin_city`, `id_destination_city`, `weekly_train`, `desserte_type`

**Sources :** `data/germany/`
- `routes.csv` — infos sur les lignes
- `trips.csv` — association route ↔ trip
- `stop_times.csv` — séquence des arrêts par trip
- `stops.csv` — métadonnées des arrêts

## 0. Imports & configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../../../data/spain")
DATA_SOURCE = "spain"

print(f"Dossier de données : {DATA_DIR.resolve()}")

Dossier de données : /home/thomas/Documents/mspr/TPRE612/data/spain


## 1. Chargement des fichiers bruts

In [2]:
routes = pd.read_csv(
    DATA_DIR / "routes.csv",
    usecols=["route_id", "route_short_name", "route_long_name", "route_type"],
    dtype=str
)

trips = pd.read_csv(
    DATA_DIR / "trips.csv",
    usecols=["route_id", "trip_id"],
    dtype=str
)

stop_times = pd.read_csv(
    DATA_DIR / "stop_times.csv",
    usecols=["trip_id", "stop_id", "stop_sequence"],
    dtype={"trip_id": str, "stop_id": str, "stop_sequence": int}
)

stops = pd.read_csv(
    DATA_DIR / "stops.csv",
    usecols=["stop_id", "stop_name", "parent_station"],
    dtype=str
)

print(f"routes     : {routes.shape}")
print(f"trips      : {trips.shape}")
print(f"stop_times : {stop_times.shape}")
print(f"stops      : {stops.shape}")

routes     : (680, 4)
trips      : (6385, 2)
stop_times : (51955, 3)
stops      : (1019, 3)


## 2. Extraction des arrêts origine / destination par trip

Pour chaque `trip_id`, on retient :
- **origin** = arrêt avec le `stop_sequence` le plus petit
- **destination** = arrêt avec le `stop_sequence` le plus grand

In [3]:
# Premier arrêt de chaque trip
origin = (
    stop_times
    .sort_values("stop_sequence")
    .groupby("trip_id", as_index=False)
    .first()[["trip_id", "stop_id"]]
    .rename(columns={"stop_id": "id_origin_city"})
)

# Dernier arrêt de chaque trip
destination = (
    stop_times
    .sort_values("stop_sequence")
    .groupby("trip_id", as_index=False)
    .last()[["trip_id", "stop_id"]]
    .rename(columns={"stop_id": "id_destination_city"})
)

od_per_trip = origin.merge(destination, on="trip_id")

# Filtrer les trips sans mouvement (origine == destination)
od_per_trip = od_per_trip[od_per_trip["id_origin_city"] != od_per_trip["id_destination_city"]]

print(f"Trips avec OD valides : {len(od_per_trip):,}")
od_per_trip.head()

Trips avec OD valides : 6,385


,trip_id,id_origin_city,id_destination_city
0,0019012026-05-20,17000,37606
1,0019012026-08-05,17000,37606
2,0019012026-10-13,17000,37606
3,0019012026-10-15,17000,37606
4,0019012026-11-03,17000,37606


## 3. Jointures : trips → routes + OD

In [4]:
# Associer chaque trip à sa route
trips_enriched = trips.merge(od_per_trip, on="trip_id", how="inner")

# Ajouter les infos de route
trips_enriched = trips_enriched.merge(routes, on="route_id", how="left")

print(f"Lignes après jointures : {len(trips_enriched):,}")
trips_enriched.head()

Lignes après jointures : 6,385


,route_id,trip_id,id_origin_city,id_destination_city,route_short_name,route_long_name,route_type
0,1700037606GL023,0019012026-05-20,17000,37606,ALVIA,NaN,2
1,1700037606GL023,0019012026-08-05,17000,37606,ALVIA,NaN,2
2,1700037606GL023,0019012026-10-13,17000,37606,ALVIA,NaN,2
3,1700037606GL023,0019012026-10-15,17000,37606,ALVIA,NaN,2
4,1700037606GL023,0019012026-11-03,17000,37606,ALVIA,NaN,2


## 4. Calcul de `weekly_train`

On agrège par `(route_id, id_origin_city, id_destination_city)` et on compte les trips distincts.
Ce comptage est un proxy du volume hebdomadaire (les données GTFS représentent typiquement une semaine type).

In [5]:
aggregated = (
    trips_enriched
    .groupby(["route_id", "id_origin_city", "id_destination_city"], as_index=False)
    .agg(weekly_train=("trip_id", "nunique"))
)

print(f"Paires OD uniques : {len(aggregated):,}")
print(f"\nDistribution weekly_train :")
print(aggregated["weekly_train"].describe())

Paires OD uniques : 680

Distribution weekly_train :
count    680.000000
mean       9.389706
std       16.805731
min        1.000000
25%        2.000000
50%        4.000000
75%       10.000000
max      252.000000
Name: weekly_train, dtype: float64


## 5. Calcul de `desserte_type`

| Condition | Valeur |
|---|---|
| `weekly_train < 7` | `Sous-desservi` |
| `7 ≤ weekly_train ≤ 56` | `Desserte Normale` |
| `weekly_train > 56` | `Bien desservi` |

In [6]:
def classify_desserte(n):
    if n < 7:
        return "Sous-desservi"
    elif n <= 56:
        return "Desserte Normale"
    else:
        return "Bien desservi"

aggregated["desserte_type"] = aggregated["weekly_train"].apply(classify_desserte)

print("Répartition desserte_type :")
print(aggregated["desserte_type"].value_counts())

Répartition desserte_type :
desserte_type
Sous-desservi       452
Desserte Normale    211
Bien desservi        17
Name: count, dtype: int64


## 6. Construction du DataFrame final

In [7]:
OUTPUT_COLS = [
    "data_source",
    "route_id",
    "id_origin_city",
    "id_destination_city",
    "weekly_train",
    "desserte_type",
]

result = aggregated.copy()
result["data_source"] = DATA_SOURCE
result = result[OUTPUT_COLS].sort_values(["route_id", "id_origin_city", "id_destination_city"]).reset_index(drop=True)

print(f"Shape finale : {result.shape}")
print(f"Colonnes     : {list(result.columns)}")
result.head(10)

Shape finale : (680, 6)
Colonnes     : ['data_source', 'route_id', 'id_origin_city', 'id_destination_city', 'weekly_train', 'desserte_type']


,data_source,route_id,id_origin_city,id_destination_city,weekly_train,desserte_type
0,spain,0100751003VRX,01007,51003,1,Sous-desservi
1,spain,0203055020VRX,02030,55020,5,Sous-desservi
2,spain,0310017000VRX,03100,17000,6,Sous-desservi
3,spain,0310050500VRX,03100,50500,2,Sous-desservi
4,spain,0310051405VRX,03100,51405,15,Desserte Normale
5,spain,0321303216AV008,03213,03216,23,Desserte Normale
6,spain,0321317000AV006,03213,17000,24,Desserte Normale
7,spain,0321317000LC112,03213,17000,7,Desserte Normale
8,spain,0321360000AV006,03213,60000,2,Sous-desservi
9,spain,0321603213AV008,03216,03213,15,Desserte Normale


## 7. Contrôles qualité

In [8]:
print("=== Valeurs nulles ===")
print(result.isnull().sum())

print("\n=== Doublons ===")
n_dup = result.duplicated(subset=["route_id", "id_origin_city", "id_destination_city"]).sum()
print(f"{n_dup} doublon(s) détecté(s)")

print("\n=== Cohérence weekly_train ===")
assert (result["weekly_train"] > 0).all(), "Des weekly_train nuls ou négatifs détectés !"
print("OK — tous les weekly_train sont > 0")

print("\n=== Valeurs desserte_type ===")
valid_types = {"Sous-desservi", "Desserte Normale", "Bien desservi"}
assert set(result["desserte_type"].unique()).issubset(valid_types)
print("OK — valeurs conformes")

=== Valeurs nulles ===
data_source            0
route_id               0
id_origin_city         0
id_destination_city    0
weekly_train           0
desserte_type          0
dtype: int64

=== Doublons ===
0 doublon(s) détecté(s)

=== Cohérence weekly_train ===
OK — tous les weekly_train sont > 0

=== Valeurs desserte_type ===
OK — valeurs conformes


## 8. Export

In [11]:
OUTPUT_PATH = Path("../../../data/output/spain_etl.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

result.to_csv(OUTPUT_PATH, index=False)
print(f"  Export: {OUTPUT_PATH.resolve()}")
print(f"  {len(result):,} lignes exportées")

  Export: /home/thomas/Documents/mspr/TPRE612/data/output/spain_etl.csv
  680 lignes exportées
